#Task 1: Prompt Engineering (LangChain Prompt Templates)

In [ ]:
!pip install langchain langchain-core --quiet

In [ ]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate


## Task 1: Replace Hardcoded Prompts

In [ ]:
explain_template = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} in simple terms for beginners"
)

output_1 = explain_template.format(topic="Machine Learning")
print(output_1)

Explain Machine Learning in simple terms for beginners



## Task 2: Multi-Input Prompt System

In [ ]:
multi_input_template = PromptTemplate(
    input_variables=["topic", "audience", "tone"],
    template="Explain {topic} for {audience} in a {tone} tone"
)

test_cases = [
    {"topic": "AI",            "audience": "beginners",  "tone": "friendly"},
    {"topic": "Python",        "audience": "kids",        "tone": "fun"},
    {"topic": "Deep Learning", "audience": "engineers",   "tone": "technical"},
]

for case in test_cases:
    prompt = multi_input_template.format(**case)
    print(f"{prompt}")

Explain AI for beginners in a friendly tone
Explain Python for kids in a fun tone
Explain Deep Learning for engineers in a technical tone



## Task 3: Prompt Variations Engine

In [ ]:
teaching_template = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} clearly step by step"
)

interview_template = PromptTemplate(
    input_variables=["topic"],
    template="Ask 3 questions about {topic}"
)

storytelling_template = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} as a story"
)


variation_templates = {
    "Teaching":     teaching_template,
    "Interview":    interview_template,
    "Storytelling": storytelling_template,
}

topic = "Machine Learning"

for style, template in variation_templates.items():
    prompt = template.format(topic=topic)
    print(f"[{style}] -> {prompt}")

[Teaching] -> Explain Machine Learning clearly step by step
[Interview] -> Ask 3 questions about Machine Learning
[Storytelling] -> Explain Machine Learning as a story



## Task 4: ChatPromptTemplate System

In [ ]:
role_descriptions = {
    "teacher":     "You are a patient teacher who explains concepts clearly with examples.",
    "interviewer": "You are a technical interviewer who asks probing questions to test understanding.",
    "motivator":   "You are an enthusiastic motivator who inspires learners to keep going.",
}

# System message controls assistant behavior; human message carries the actual request
system_template = SystemMessagePromptTemplate.from_template("{role_description}")
human_template  = HumanMessagePromptTemplate.from_template("Please help me understand {topic}.")

chat_prompt = ChatPromptTemplate.from_messages([system_template, human_template])

def build_chat_prompt(role, topic):
    """Looks up the role description, injects it into the chat prompt, returns formatted messages."""
    role_description = role_descriptions[role]
    return chat_prompt.format_messages(role_description=role_description, topic=topic)

role  = "teacher"
topic = "Neural Networks"
messages = build_chat_prompt(role, topic)

for msg in messages:
    print(f"[{msg.type.upper()}]: {msg.content}")

print()
for role in ["interviewer", "motivator"]:
    msgs = build_chat_prompt(role, topic)
    for msg in msgs:
        print(f"[{msg.type.upper()} | {role}]: {msg.content}")
    print()

  [SYSTEM]: You are a patient teacher who explains concepts clearly with examples.
  [HUMAN]: Please help me understand Neural Networks.

[SYSTEM | interviewer]: You are a technical interviewer who asks probing questions to test understanding.
[HUMAN | interviewer]: Please help me understand Neural Networks.

[SYSTEM | motivator]: You are an enthusiastic motivator who inspires learners to keep going.
[HUMAN | motivator]: Please help me understand Neural Networks.




## Task 5: Input Validation Layer

In [ ]:
VALID_AUDIENCES = ["beginner", "intermediate", "expert"]
VALID_TONES     = ["formal", "casual", "fun"]

DEFAULT_AUDIENCE = "beginner"
DEFAULT_TONE     = "casual"

def validate_inputs(audience, tone):
    """
    Checks audience and tone against allowed values.
    Bad inputs fall back to defaults instead of crashing -- intentional choice
    since real users shouldn't see uncaught errors for a typo.
    Returns (audience, tone).
    """
    validated_audience = audience if audience in VALID_AUDIENCES else DEFAULT_AUDIENCE
    validated_tone=tone if tone in VALID_TONES else DEFAULT_TONE

    if validated_audience != audience:
        print(f" [Warning] Invalid audience '{audience}'. Defaulting to '{DEFAULT_AUDIENCE}'.")
    if validated_tone != tone:
        print(f" [Warning] Invalid tone '{tone}'. Defaulting to '{DEFAULT_TONE}'.")

    return validated_audience, validated_tone


a, t = validate_inputs("beginner", "formal")
print(f"  Valid -> audience='{a}', tone='{t}'")

a, t = validate_inputs("child", "fun")         # bad audience
print(f"  Fixed -> audience='{a}', tone='{t}'")

a, t = validate_inputs("expert", "aggressive")  # bad tone
print(f"  Fixed -> audience='{a}', tone='{t}'")

a, t = validate_inputs("anyone", "serious")     # both bad
print(f"  Fixed -> audience='{a}', tone='{t}'")

  Valid -> audience='beginner', tone='formal'
 [Warning] Invalid audience 'child'. Defaulting to 'beginner'.
  Fixed -> audience='beginner', tone='fun'
 [Warning] Invalid tone 'aggressive'. Defaulting to 'casual'.
  Fixed -> audience='expert', tone='casual'
 [Warning] Invalid audience 'anyone'. Defaulting to 'beginner'.
 [Warning] Invalid tone 'serious'. Defaulting to 'casual'.
  Fixed -> audience='beginner', tone='casual'



## Task 6: Prompt Generator App

In [ ]:
style_templates = {
    "teaching": PromptTemplate(
        input_variables=["topic", "audience", "tone"],
        template="Explain {topic} to {audience} in a {tone} teaching style, step by step"
    ),
    "interview": PromptTemplate(
        input_variables=["topic", "audience", "tone"],
        template="Generate 3 {tone} interview questions about {topic} for a {audience} candidate"
    ),
    "storytelling": PromptTemplate(
        input_variables=["topic", "audience", "tone"],
        template="Explain {topic} to {audience} through a {tone} story"
    ),
}

VALID_STYLES = list(style_templates.keys())

def generate_prompt(topic, audience, tone, style):
    """
    Full pipeline: validate inputs -> pick template by style -> format and return.
    Style raises ValueError rather than defaulting -- there's no sensible fallback
    for an unknown style, unlike audience or tone.
    """
    audience, tone = validate_inputs(audience, tone)

    if style not in VALID_STYLES:
        raise ValueError(f"Unknown style '{style}'. Options: {VALID_STYLES}")

    return style_templates[style].format(topic=topic, audience=audience, tone=tone)

print(generate_prompt("Neural Networks", "beginners",    "fun",    "storytelling"))
print(generate_prompt("Python",          "intermediate", "formal", "interview"))
print(generate_prompt("Data Science",    "expert",       "casual", "teaching"))

print("\nWith invalid audience:")
print(generate_prompt("AI Ethics", "anyone", "fun", "teaching"))

 [Warning] Invalid audience 'beginners'. Defaulting to 'beginner'.
Explain Neural Networks to beginner through a fun story
Generate 3 formal interview questions about Python for a intermediate candidate
Explain Data Science to expert in a casual teaching style, step by step

With invalid audience:
 [Warning] Invalid audience 'anyone'. Defaulting to 'beginner'.
Explain AI Ethics to beginner in a fun teaching style, step by step



## Task 7: Template Reusability Test

In [ ]:
reusable_template = PromptTemplate(
    input_variables=["topic", "audience", "tone"],
    template="Explain {topic} for {audience} in a {tone} tone"
)

reusability_test_cases = [
    {"topic": "Machine Learning","audience": "beginners","tone": "friendly"},
    {"topic": "Cybersecurity","audience": "intermediate","tone": "serious"},
    {"topic": "Neural Networks","audience": "engineers","tone": "technical"},
    {"topic": "Cloud Computing","audience": "managers","tone": "formal"},
    {"topic": "Natural Language Processing","audience": "students","tone": "casual"},
]

for i, case in enumerate(reusability_test_cases, start=1):
    prompt = reusable_template.format(**case)
    print(f"Case {i}: {prompt}")

Case 1: Explain Machine Learning for beginners in a friendly tone
Case 2: Explain Cybersecurity for intermediate in a serious tone
Case 3: Explain Neural Networks for engineers in a technical tone
Case 4: Explain Cloud Computing for managers in a formal tone
Case 5: Explain Natural Language Processing for students in a casual tone
